In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, concat, lit, lpad, year, month, day, dayofweek, explode, sequence, expr

# Znalezienie min i max roku
df_date = spark.sql("SELECT MIN(year) as min_date, MAX(year) as max_date FROM flight_data_bronze.flights")

df_todate = df_date.select(
    to_date(concat(
        col("min_date"),
        lit("-01-01"),
        ),"yyyy-MM-dd").alias("min_date"),
    to_date(concat(
        col("max_date"),
        lit("-12-31"),
        ),"yyyy-MM-dd").alias("max_date")
)

# Przygotowanie tabeli
df_dates = df_todate.select(
    explode(sequence(col("min_date"), col("max_date"), expr("interval 1 day"))).alias("date_key"),
    year(col("date_key")).alias("year"),
    month(col("date_key")).alias("month"),
    day(col("date_key")).alias("day"),
    dayofweek(col("date_key")).alias("day_of_week")
)

# Zapisanie tabeli
df_dates.createOrReplaceTempView("dates")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS flight_data_silver.dimdates(
  date_key DATE,
  year int,
  month int,
  day int,
  day_of_week int
);

MERGE INTO flight_data_silver.dimdates AS dima
USING dates AS a
ON dima.date_key = a.date_key
WHEN NOT MATCHED
THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
365,0,0,365


In [0]:
%sql
SELECT * FROM flight_data_silver.dimdates

date_key,year,month,day,day_of_week
2015-01-01,2015,1,1,5
2015-01-02,2015,1,2,6
2015-01-03,2015,1,3,7
2015-01-04,2015,1,4,1
2015-01-05,2015,1,5,2
2015-01-06,2015,1,6,3
2015-01-07,2015,1,7,4
2015-01-08,2015,1,8,5
2015-01-09,2015,1,9,6
2015-01-10,2015,1,10,7
